# 01.4 · Blackboard 模式（共享状态 + 事件）

> 多 Agent 通过共享黑板异步协作。


In [3]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

✅ 模拟工具加载完毕


---
## Part 4 · Blackboard 模式（共享状态 + 事件）

多个 Agent 通过**共享黑板**（字典 + `threading.Event`）协作，无需直接互调。

### 架构总览

```mermaid
flowchart TB
    subgraph Agents["独立 Agent 线程（互不调用）"]
        IA["IngestAgent"]
        RA["RetrieverAgent"]
        RR["RerankerAgent"]
        GA["GeneratorAgent"]
    end

    BB[("🗂 Blackboard<br/>_data + _events + _lock")]

    IA -->|"write query, chunks"| BB
    BB -->|"event: chunks"| RA
    RA -->|"write hits"| BB
    BB -->|"event: hits"| RR
    RR -->|"write topk"| BB
    BB -->|"event: topk"| GA
    GA -->|"write answer"| BB
```

In [4]:
# ── Blackboard 实现 ──────────────────────────────────────────────────────────
class Blackboard:
    def __init__(self):
        self._data: dict[str, Any] = {}
        self._events: dict[str, threading.Event] = defaultdict(threading.Event)
        self._lock = threading.Lock()

    def write(self, key: str, value: Any):
        with self._lock:
            self._data[key] = value
        print(f"  📋 Blackboard.write({key!r})")
        self._events[key].set()          # 通知订阅者

    def read(self, key: str) -> Any:
        with self._lock:
            return self._data.get(key)

    def wait_for(self, key: str, timeout: float = 5.0) -> bool:
        """阻塞直到 key 被写入"""
        return self._events[key].wait(timeout=timeout)


# ── 各 Agent 定义（每个都是独立线程）────────────────────────────────────────
def ingest_agent(bb: Blackboard, query: str):
    print("  [Ingest] 开始分片...")
    time.sleep(0.1)
    bb.write("query", query)
    bb.write("chunks", [{"text": t} for t in CORPUS[:4]])

def retriever_agent(bb: Blackboard):
    bb.wait_for("chunks")
    query  = bb.read("query")
    chunks = bb.read("chunks")
    print("  [Retriever] 检索中...")
    time.sleep(0.3)
    hits = [
        {"text": c["text"], "score": round(random.uniform(0.5, 1.0), 3)}
        for c in sorted(chunks, key=lambda _: random.random())[:3]
    ]
    bb.write("hits", hits)

def reranker_agent(bb: Blackboard):
    bb.wait_for("hits")
    query = bb.read("query")
    hits  = bb.read("hits")
    print("  [Reranker] 重排中...")
    time.sleep(0.2)
    topk = fake_rerank(query, hits, top_k=2, latency=0)
    bb.write("topk", topk)

def generator_agent(bb: Blackboard):
    ok = bb.wait_for("topk", timeout=5.0)
    query = bb.read("query")
    topk  = bb.read("topk") or []
    print("  [Generator] 生成中...")
    answer = fake_generate(query, topk, latency=0.5)
    bb.write("answer", answer)


# ── 运行：启动所有 Agent，让它们通过黑板协作 ─────────────────────────────────
bb = Blackboard()
t0 = time.perf_counter()

threads = [
    threading.Thread(target=ingest_agent,    args=(bb, "什么是 Blackboard 模式？")),
    threading.Thread(target=retriever_agent, args=(bb,)),
    threading.Thread(target=reranker_agent,  args=(bb,)),
    threading.Thread(target=generator_agent, args=(bb,)),
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\n⏱ 总耗时：{round(time.perf_counter()-t0, 3)}s")
print(f"📝 最终答案：{bb.read('answer')[:80]}...")

  [Ingest] 开始分片...
  📋 Blackboard.write('query')
  📋 Blackboard.write('chunks')
  [Retriever] 检索中...
  📋 Blackboard.write('hits')
  [Reranker] 重排中...
  📋 Blackboard.write('topk')
  [Generator] 生成中...
  📋 Blackboard.write('answer')

⏱ 总耗时：1.132s
📝 最终答案：[答案] 关于「什么是 Blackboard 模式？」：向量数据库支持语义相似度搜索。 RAG 通过检索外部文档来扩充 LLM 的上下文。......


> **关键观察**：
> - 各 Agent **不互相调用**，只通过黑板读写
> - `wait_for` 实现事件驱动，无需轮询
> - 新增一个 Agent 只需监听相应字段，**不修改任何已有代码**

### vs Pipeline：通信方式对比

```mermaid
flowchart LR
    subgraph PL["Pipeline（01.2）— 点对点传 job"]
        direction LR
        O1["Orchestrator"] --> Q1["q_retrieve"]
        Q1 --> W1["Retriever"]
        W1 --> Q2["q_rerank"]
        Q2 --> W2["Reranker"]
        W2 --> Q3["q_generate"]
        Q3 --> W3["Generator"]
        W3 --> R1["results"]
    end

    subgraph BB["Blackboard（01.4）— 共享状态 + 事件"]
        direction TB
        BB2[("Blackboard")]
        A1["Ingest"] <-->|"write / wait"| BB2
        A2["Retriever"] <-->|"write / wait"| BB2
        A3["Reranker"] <-->|"write / wait"| BB2
        A4["Generator"] <-->|"write / wait"| BB2
    end
```

| 维度 | Pipeline | Blackboard |
|------|----------|------------|
| 通信 | 队列传递整条 job | 黑板按 key 读写 |
| 耦合 | Worker 知道上下游队列 | Agent 只知道字段名 |
| 扩展 | 插阶段要改队列链路 | 新 Agent 订阅新 key 即可 |
| 适用 | 固定线性流水线 | 迭代协作、长任务、多角色协商 |

## 📖 讲义

# Part 4
## 三个动手示例

---

# 示例概览

| 示例 | 模式 | 技术栈 | 难度 |
|------|------|--------|------|
| **示例 1** | Pipeline | Redis + Python Workers | ⭐ 入门 |
| **示例 2** | Hub-and-Spoke | FastAPI + HTTP | ⭐⭐ 进阶 |
| **示例 3** | Blackboard | DB + Pub-Sub | ⭐⭐⭐ 高级 |

> 每个示例均可在本地独立运行，课后建议按顺序动手实现

---

# 示例 1：Redis Pipeline（架构）

```
用户请求
   │
   ▼
Orchestrator ──push──▶ jobs:retriever
                              │
                        RetrieverWorker
                              │ push
                              ▼
                        jobs:reranker
                              │
                        RerankerWorker
                              │ push
                              ▼
                        jobs:generator
                              │
                        GeneratorWorker ──write──▶ results:{job_id}
                                                        │
                                                        ▼
                                                  API 轮询取结果
```

---

# 示例 1：核心代码

**Orchestrator — 提交任务**
```python
# orchestrator.py
import redis, json, uuid

r = redis.Redis()

def submit_query(query: str) -> str:
    job_id = str(uuid.uuid4())
    job = {"request_id": job_id, "query": query, "stage": "retriever"}
    r.lpush("jobs:retriever", json.dumps(job))
    return job_id
```

**Retriever Worker — 消费 & 转发**
```python
# retriever_worker.py
while True:
    _, data = r.brpop("jobs:retriever")   # blocking pop
    job = json.loads(data)

    # 幂等检查
    if r.get(f"done:retriever:{job['request_id']}"):
        continue

    job["hits"] = do_retrieval(job["query"], top_k=20)
    job["stage"] = "reranker"
    r.lpush("jobs:reranker", json.dumps(job))
    r.setex(f"done:retriever:{job['request_id']}", 3600, 1)
```

---

# 示例 1：Reranker & Generator

**Reranker Worker**
```python
# reranker_worker.py
while True:
    _, data = r.brpop("jobs:reranker")
    job = json.loads(data)
    job["topk"] = cross_encoder_rerank(job["query"], job["hits"], top_k=5)
    job["stage"] = "generator"
    r.lpush("jobs:generator", json.dumps(job))
```

**Generator Worker**
```python
# generator_worker.py
while True:
    _, data = r.brpop("jobs:generator")
    job = json.loads(data)
    context = "\n".join([h["text"] for h in job["topk"]])
    answer = llm_generate(question=job["query"], context=context)
    r.setex(f"result:{job['request_id']}", 3600, json.dumps({"answer": answer}))
```

> **运行方式**：分别在 3 个终端启动三个 Worker，再调用 `submit_query()`

---

# 示例 2：HTTP Orchestrator（架构）

```
用户
 │ POST /query
 ▼
┌─────────────────────────────────┐
│         Orchestrator            │
│  1. POST /retrieve → hits       │
│  2. POST /rerank   → topk       │
│  3. POST /generate → answer     │
│  4. POST /verify   → verdict    │
└─────────────────────────────────┘
     │         │         │         │
 :8001      :8002     :8003     :8004
Retriever  Reranker Generator Verifier
```

---

# 示例 2：Orchestrator 代码

```python
# orchestrator_http.py
import httpx
from fastapi import FastAPI

app = FastAPI()
SERVICES = {
    "retriever": "http://localhost:8001",
    "reranker":  "http://localhost:8002",
    "generator": "http://localhost:8003",
    "verifier":  "http://localhost:8004",
}

@app.post("/query")
async def handle_query(payload: dict):
    q = payload["query"]
    async with httpx.AsyncClient(timeout=30) as client:
        hits    = (await client.post(f"{SERVICES['retriever']}/retrieve",
                                      json={"query": q})).json()["hits"]
        topk    = (await client.post(f"{SERVICES['reranker']}/rerank",
                                      json={"query": q, "candidates": hits})).json()["topk"]
        answer  = (await client.post(f"{SERVICES['generator']}/generate",
                                      json={"question": q, "context": topk})).json()["answer"]
        verdict = (await client.post(f"{SERVICES['verifier']}/check",
                                      json={"answer": answer, "context": topk})).json()
    return {"answer": answer, "verdict": verdict}
```

> 每个微服务独立用 FastAPI 实现，用 Docker Compose 一键启动

---

# 示例 3：Blackboard 模式

**核心思路**：共享数据库作为"黑板"，Agent 通过事件感知变化

```python
# blackboard.py — 简化示意
import redis

r = redis.Redis()

class Blackboard:
    def write(self, task_id: str, field: str, value):
        """写入字段并发布变更事件"""
        r.hset(f"task:{task_id}", field, json.dumps(value))
        r.publish(f"task:{task_id}:events", field)

    def read(self, task_id: str, field: str):
        raw = r.hget(f"task:{task_id}", field)
        return json.loads(raw) if raw else None

    def subscribe(self, task_id: str):
        """订阅该任务的所有变更"""
        pubsub = r.pubsub()
        pubsub.subscribe(f"task:{task_id}:events")
        return pubsub
```

**流程**：Ingest Agent 写 `chunks` → Retriever 订阅 `new_query` 事件并写 `hits` → Reranker 订阅 `hits_ready` 并写 `topk` → Generator 读取 `topk` 生成并写 `answer`

---

---
**导航**：[← 01.3 · Hub-and-Spoke 模式](./01.3_hub_and_spoke.ipynb) · [📋 课程索引](./01_multi_agent_arch_demo.ipynb) · [01.5 · 幂等设计 →](./01.5_idempotency.ipynb)
